# HGCTM Revision — Prior and Lithology-Class Sensitivity at Fixed K = 7

This notebook addresses the reviewers' concern that the reported lithology-to-age effect ratio may be influenced by the model's wider lithology prior.

It fits the final sparse M7b HGCTM under several prior specifications while keeping the following fixed:

- the same 1,237 valid-coordinate deposits;
- the same 35 mineral-family count matrix;
- K = 7;
- the same age categories;
- the same Macrostrat, GLiM, and combined lithology assignments;
- the same LDA warm start;
- the same optimization settings;
- the same random seeds.

## Prior settings

1. **Original:** lithology scale 0.30, age scale 0.15.
2. **Equal-low:** lithology scale 0.15, age scale 0.15.
3. **Equal-high:** lithology scale 0.30, age scale 0.30.
4. **Reversed:** lithology scale 0.15, age scale 0.30.
5. **Learned equal hyperpriors:** both scales are inferred from the data under identical HalfNormal(0.5) hyperpriors.

Macrostrat and GLiM are fitted under all five settings. The combined source is fitted under the original, equal-high, and learned-scale settings.

## Rare-class handling

The original raw labels are preserved. For model fitting only:

- the single GLiM `evaporite` record is merged into `other`;
- no empty evaporite parameter is created;
- `unknown` remains a distinct missingness category;
- rare and zero-count class warnings are exported;
- the primary effect ratio uses class-frequency-weighted deviations;
- additional unweighted and strict-resolved ratios are also reported.

## Important interpretation

The learned-scale model preserves the original age-weight parameter omega_a. Therefore, the learned age scale and omega_a should be interpreted together through the **effective age deviation**, not separately as perfectly identifiable quantities.

Original notebook SHA-256: `6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b`

## Required Kaggle inputs

Attach:

1. `Hgctm_stage0.zip` or the extracted Stage 0 dataset containing:
   - `X_family_copper.csv`
   - `covariates.csv`

2. `HGCTM_Phase1B_GLiM_Audit_Outputs.zip` or its extracted files containing:
   - `covariates_lithology_A_macrostrat_original.csv`
   - `covariates_lithology_B_glim_only.csv`
   - `covariates_lithology_C_macrostrat_glim_combined.csv`

Optional for resuming:

3. A previous output ZIP from this notebook. Completed run files are detected and skipped.

No external API requests are made.

In [ ]:
# Install only missing packages.

import importlib.util
import subprocess
import sys

required = {
    "pyro": "pyro-ppl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_prior_lithology_sensitivity"
RUNS_DIR = OUT / "runs"
FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables"

for folder in [OUT, RUNS_DIR, FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------- EXECUTION MODE ----------------
#
# quick:
#   One seed, 400 steps. Only confirms that the notebook runs.
# screening:
#   One seed, 2,500 steps for all 13 configurations.
# full:
#   Three seeds, 5,000 steps for all 13 configurations.
#
RUN_MODE = "full"

K = 7
MIN_MINERALS = 3
MIN_DEPOSITS_PER_FAMILY = 10
EXPECTED_MODEL_N = 1335
EXPECTED_INVALID_COORDINATES = 98
EXPECTED_COMMON_N = 1237
EXPECTED_F = 35

PRIMARY_SEED = 42
FULL_SEEDS = [42, 123, 7]

if RUN_MODE == "quick":
    ACTIVE_SEEDS = [42]
    N_STEPS = 400
    ANNEAL_END = 120
    PRINT_EVERY = 100
elif RUN_MODE == "screening":
    ACTIVE_SEEDS = [42]
    N_STEPS = 2500
    ANNEAL_END = 750
    PRINT_EVERY = 500
elif RUN_MODE == "full":
    ACTIVE_SEEDS = FULL_SEEDS
    N_STEPS = 5000
    ANNEAL_END = 1500
    PRINT_EVERY = 500
else:
    raise ValueError("RUN_MODE must be quick, screening, or full.")

LR = 0.005
RARE_CLASS_THRESHOLD = 20
SIGMA_HYPERPRIOR_SCALE = 0.5

AGE_LEVELS = [
    "Cenozoic",
    "Mesozoic",
    "Paleozoic",
    "Precambrian",
    "Unknown",
]

SOURCE_SPECS = {
    "Macrostrat": "lithology_A_class",
    "GLiM": "lithology_B_class",
    "Combined": "lithology_C_class",
}

PRIOR_SETTINGS = {
    "original": {
        "learn_scales": False,
        "sigma_litho": 0.30,
        "sigma_age": 0.15,
        "description": "Submitted ordered-shrinkage setting",
    },
    "equal_low": {
        "learn_scales": False,
        "sigma_litho": 0.15,
        "sigma_age": 0.15,
        "description": "Equal conservative prior scales",
    },
    "equal_high": {
        "learn_scales": False,
        "sigma_litho": 0.30,
        "sigma_age": 0.30,
        "description": "Equal wider prior scales",
    },
    "reversed": {
        "learn_scales": False,
        "sigma_litho": 0.15,
        "sigma_age": 0.30,
        "description": "Age given wider prior scale",
    },
    "learned_equal_hyperprior": {
        "learn_scales": True,
        "sigma_litho": None,
        "sigma_age": None,
        "description": "Both scales learned under identical HalfNormal(0.5) priors",
    },
}

RUN_PLAN = {
    "Macrostrat": [
        "original",
        "equal_low",
        "equal_high",
        "reversed",
        "learned_equal_hyperprior",
    ],
    "GLiM": [
        "original",
        "equal_low",
        "equal_high",
        "reversed",
        "learned_equal_hyperprior",
    ],
    "Combined": [
        "original",
        "equal_high",
        "learned_equal_hyperprior",
    ],
}

print("Run mode:", RUN_MODE)
print("Seeds:", ACTIVE_SEEDS)
print("Steps:", N_STEPS)
print("Configurations:", sum(len(v) for v in RUN_PLAN.values()))
print("Total planned fits:", sum(len(v) for v in RUN_PLAN.values()) * len(ACTIVE_SEEDS))


## 1. Locate inputs and optional resume files

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
    return target


def find_folder_with(required_names, label):
    required_names = set(required_names)
    roots = [Path("/kaggle/input"), WORK]

    for root in roots:
        if not root.exists():
            continue
        for first_name in required_names:
            for path in root.rglob(first_name):
                folder = path.parent
                present = {
                    item.name for item in folder.iterdir()
                    if item.is_file()
                }
                if required_names.issubset(present):
                    return folder

    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name for name in archive.namelist()
                    }
                    if required_names.issubset(basenames):
                        safe_label = re.sub(
                            r"[^a-z0-9]+", "_", label.lower()
                        ).strip("_")
                        return extract_zip_once(
                            zip_path,
                            WORK / f"{safe_label}_extracted",
                        )
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        f"Could not find {label}. Required files: "
        f"{sorted(required_names)}"
    )


STAGE0 = find_folder_with(
    {"X_family_copper.csv", "covariates.csv"},
    "HGCTM Stage 0",
)

PHASE1B = find_folder_with(
    {
        "covariates_lithology_A_macrostrat_original.csv",
        "covariates_lithology_B_glim_only.csv",
        "covariates_lithology_C_macrostrat_glim_combined.csv",
    },
    "HGCTM Phase 1B",
)

print("Stage 0:", STAGE0)
print("Phase 1B:", PHASE1B)


In [ ]:
def import_resume_runs():
    imported = 0
    roots = [Path("/kaggle/input")]

    for root in roots:
        if not root.exists():
            continue

        # Extract prior output archives if present.
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    names = archive.namelist()
                    if any(
                        name.endswith("run_summary.json")
                        and "/runs/" in f"/{name}"
                        for name in names
                    ):
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:12]
                        target = WORK / f"resume_{digest}"
                        extract_zip_once(zip_path, target)
            except zipfile.BadZipFile:
                continue

        # Copy completed run artifacts from any extracted or direct folder.
        for summary_path in root.rglob("*_run_summary.json"):
            source_run_dir = summary_path.parent
            run_stem = summary_path.name.replace(
                "_run_summary.json", ""
            )
            destination_summary = (
                RUNS_DIR / f"{run_stem}_run_summary.json"
            )

            if destination_summary.exists():
                continue

            for source_file in source_run_dir.glob(f"{run_stem}*"):
                if source_file.is_file():
                    shutil.copy2(
                        source_file,
                        RUNS_DIR / source_file.name,
                    )
            imported += 1

    # Also scan extracted resume folders under working.
    for root in WORK.glob("resume_*"):
        for summary_path in root.rglob("*_run_summary.json"):
            run_stem = summary_path.name.replace(
                "_run_summary.json", ""
            )
            destination_summary = (
                RUNS_DIR / f"{run_stem}_run_summary.json"
            )
            if destination_summary.exists():
                continue
            for source_file in summary_path.parent.glob(f"{run_stem}*"):
                if source_file.is_file():
                    shutil.copy2(
                        source_file,
                        RUNS_DIR / source_file.name,
                    )
            imported += 1

    return imported


imported_resume_runs = import_resume_runs()
print("Imported completed resume runs:", imported_resume_runs)


## 2. Load and normalize the frozen data

In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def canonicalize_index(frame, label):
    frame = frame.copy()
    frame.index = [canonical_id(value) for value in frame.index]
    frame.index.name = "Mindat_id"

    if frame.index.isna().any():
        raise ValueError(f"{label} contains missing Mindat IDs.")
    if frame.index.duplicated().any():
        duplicates = frame.index[
            frame.index.duplicated()
        ].tolist()
        raise ValueError(
            f"{label} contains duplicated IDs: {duplicates[:10]}"
        )
    return frame


def canonicalize_id_column(frame, label):
    frame = frame.copy()

    if "Mindat_id" not in frame.columns:
        raise KeyError(f"{label} has no Mindat_id column.")

    frame["Mindat_id"] = frame["Mindat_id"].map(canonical_id)

    if frame["Mindat_id"].isna().any():
        raise ValueError(f"{label} contains missing Mindat IDs.")
    if frame["Mindat_id"].duplicated().any():
        duplicates = frame.loc[
            frame["Mindat_id"].duplicated(keep=False),
            "Mindat_id",
        ].tolist()
        raise ValueError(
            f"{label} contains duplicated IDs: {duplicates[:10]}"
        )

    return frame


X_raw = canonicalize_index(
    pd.read_csv(
        STAGE0 / "X_family_copper.csv",
        index_col=0,
    ),
    "X_family_copper",
)

cov_stage0 = canonicalize_index(
    pd.read_csv(
        STAGE0 / "covariates.csv",
        index_col=0,
    ),
    "covariates",
)

source_a = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_A_macrostrat_original.csv"
    ),
    "Macrostrat covariates",
)

source_b = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_B_glim_only.csv"
    ),
    "GLiM covariates",
)

source_c = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_C_macrostrat_glim_combined.csv"
    ),
    "Combined covariates",
)

print("X raw:", X_raw.shape)
print("Stage 0 covariates:", cov_stage0.shape)
print("Source tables:", source_a.shape, source_b.shape, source_c.shape)


## 3. Reconstruct the original 1,335-deposit matrix and common 1,237-deposit cohort

The family filter is calculated on the original 1,335-deposit model set before the invalid-coordinate records are excluded. This preserves the submitted 35-family vocabulary.

In [ ]:
model_mask = X_raw.sum(axis=1) >= MIN_MINERALS
X_pre_family = X_raw.loc[model_mask].copy()

family_presence = (X_pre_family > 0).sum(axis=0)
family_keep = family_presence[
    family_presence >= MIN_DEPOSITS_PER_FAMILY
].index.tolist()

X_model = X_pre_family[family_keep].copy()

model_ids = [
    mindat_id
    for mindat_id in X_model.index
    if mindat_id in cov_stage0.index
]

X_model = X_model.loc[model_ids]
cov_model = cov_stage0.loc[model_ids]

if X_model.shape != (EXPECTED_MODEL_N, EXPECTED_F):
    raise ValueError(
        f"Expected model matrix "
        f"({EXPECTED_MODEL_N}, {EXPECTED_F}), "
        f"found {X_model.shape}."
    )

print("Original modelling matrix:", X_model.shape)


In [ ]:
def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False

    try:
        lat = float(lat)
        lon = float(lon)
    except Exception:
        return False

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return False

    if abs(lat) < 1e-12 and abs(lon) < 1e-12:
        return False

    return True


a_small = source_a[
    [
        "Mindat_id",
        "Latitude",
        "Longitude",
        "lithology_A_class",
        "lithology_A_status",
    ]
].copy()

b_small = source_b[
    [
        "Mindat_id",
        "lithology_B_class",
        "lithology_B_status",
        "glim_level1_code",
        "glim_level1_description",
        "lithology_B_confidence",
    ]
].copy()

c_small = source_c[
    [
        "Mindat_id",
        "lithology_C_class",
        "lithology_C_status",
        "lithology_C_source",
        "lithology_C_confidence",
        "macrostrat_class",
        "macrostrat_status",
        "glim_model_class",
        "glim_status",
        "agreement_status",
    ]
].copy()

lithology_table = (
    a_small
    .merge(
        b_small,
        on="Mindat_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        c_small,
        on="Mindat_id",
        how="inner",
        validate="one_to_one",
    )
)

lithology_table["coordinate_valid"] = (
    lithology_table.apply(
        lambda row: valid_coordinate(
            row["Latitude"],
            row["Longitude"],
        ),
        axis=1,
    )
)

excluded_invalid = lithology_table[
    ~lithology_table["coordinate_valid"]
].copy()

valid_ids_set = set(
    lithology_table.loc[
        lithology_table["coordinate_valid"],
        "Mindat_id",
    ]
)

common_ids = [
    mindat_id
    for mindat_id in X_model.index
    if mindat_id in valid_ids_set
]

X_common = X_model.loc[common_ids].copy()
cov_common = cov_model.loc[common_ids].copy()

lithology_common = (
    lithology_table
    .set_index("Mindat_id")
    .loc[common_ids]
    .copy()
)

if len(excluded_invalid) != EXPECTED_INVALID_COORDINATES:
    raise ValueError(
        f"Expected {EXPECTED_INVALID_COORDINATES} invalid coordinates, "
        f"found {len(excluded_invalid)}."
    )

if X_common.shape != (EXPECTED_COMMON_N, EXPECTED_F):
    raise ValueError(
        f"Expected common matrix "
        f"({EXPECTED_COMMON_N}, {EXPECTED_F}), "
        f"found {X_common.shape}."
    )

print("Excluded invalid coordinates:", len(excluded_invalid))
print("Common cohort:", X_common.shape)

excluded_invalid.to_csv(
    TABLE_DIR / "excluded_invalid_coordinate_records.csv",
    index=False,
)


## 4. Prepare lithology classes and preserve raw labels

The raw source classes are retained in the audit table.

For fitting only, `evaporite` is merged into `other`. This prevents a single GLiM observation from creating a standalone poorly identified class.

In [ ]:
cohort = cov_common[
    [
        "Deposit_type",
        "age_category",
        "Latitude",
        "Longitude",
        "age_midpoint_ma",
    ]
].copy()

cohort["Mindat_id"] = cohort.index

for source_name, raw_column in SOURCE_SPECS.items():
    raw_values = (
        lithology_common[raw_column]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    cohort[f"{source_name}_raw_lithology"] = raw_values
    cohort[f"{source_name}_evaporite_merged"] = (
        raw_values.eq("evaporite")
    )
    cohort[f"{source_name}_model_lithology"] = (
        raw_values.replace({"evaporite": "other"})
    )

cohort = cohort.reset_index(drop=True)

model_lithology_columns = {
    source_name: f"{source_name}_model_lithology"
    for source_name in SOURCE_SPECS
}

all_model_levels = sorted(
    set().union(
        *[
            set(cohort[column].dropna().unique())
            for column in model_lithology_columns.values()
        ]
    )
)

print("Model lithology levels:", all_model_levels)

for source_name, column in model_lithology_columns.items():
    counts = cohort[column].value_counts()
    print(f"\n{source_name}:")
    print(counts.to_string())

    if "evaporite" in counts.index:
        raise ValueError(
            f"Evaporite was not merged correctly for {source_name}."
        )

cohort.to_csv(
    TABLE_DIR / "common_cohort_with_raw_and_model_lithology.csv",
    index=False,
)
X_common.to_csv(
    TABLE_DIR / "common_cohort_X_family_copper.csv"
)


In [ ]:
class_count_rows = []
class_warning_rows = []

for source_name, column in model_lithology_columns.items():
    counts = (
        cohort[column]
        .value_counts()
        .reindex(all_model_levels, fill_value=0)
    )

    for lithology_class, count in counts.items():
        class_count_rows.append({
            "source": source_name,
            "lithology_class": lithology_class,
            "count": int(count),
            "percent": 100 * count / len(cohort),
        })

        if count == 0:
            warning = "zero_count"
        elif count < RARE_CLASS_THRESHOLD:
            warning = "rare_class"
        else:
            warning = "adequate_count"

        class_warning_rows.append({
            "source": source_name,
            "lithology_class": lithology_class,
            "count": int(count),
            "threshold": RARE_CLASS_THRESHOLD,
            "warning": warning,
        })

class_counts = pd.DataFrame(class_count_rows)
class_warnings = pd.DataFrame(class_warning_rows)

class_counts.to_csv(
    TABLE_DIR / "lithology_class_counts_after_preparation.csv",
    index=False,
)
class_warnings.to_csv(
    TABLE_DIR / "lithology_class_warnings.csv",
    index=False,
)

print(class_counts.pivot(
    index="lithology_class",
    columns="source",
    values="count",
).fillna(0).astype(int).to_string())

print("\nRare/zero warnings:")
print(
    class_warnings[
        class_warnings["warning"].ne("adequate_count")
    ].to_string(index=False)
)


## 5. Fixed encodings and common LDA warm start

In [ ]:
lithology_to_index = {
    label: index
    for index, label in enumerate(all_model_levels)
}
age_to_index = {
    label: index
    for index, label in enumerate(AGE_LEVELS)
}

age_values = (
    cohort["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

unexpected_age = set(age_values.unique()) - set(AGE_LEVELS)
if unexpected_age:
    raise ValueError(
        f"Unexpected age labels: {sorted(unexpected_age)}"
    )

X_np = X_common.to_numpy(dtype=np.float32)
X_tensor = torch.tensor(X_np, dtype=torch.float32)

age_idx_np = age_values.map(
    age_to_index
).to_numpy(dtype=np.int64)
age_tensor = torch.tensor(
    age_idx_np,
    dtype=torch.long,
)

source_lithology_idx = {}
source_lithology_tensors = {}

for source_name, column in model_lithology_columns.items():
    indices = cohort[column].map(
        lithology_to_index
    ).to_numpy(dtype=np.int64)

    source_lithology_idx[source_name] = indices
    source_lithology_tensors[source_name] = torch.tensor(
        indices,
        dtype=torch.long,
    )

N, F = X_np.shape
L = len(all_model_levels)
A = len(AGE_LEVELS)

print(f"N={N}, F={F}, K={K}, L={L}, A={A}")
print("Lithology levels:", all_model_levels)
print("Age levels:", AGE_LEVELS)


In [ ]:
lda_m1 = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)

lda_m1.fit(X_np.astype(np.float64))

phi_m1 = (
    lda_m1.components_
    / lda_m1.components_.sum(axis=1, keepdims=True)
)
theta_m1 = lda_m1.transform(
    X_np.astype(np.float64)
)

np.save(OUT / "phi_m1_common_cohort.npy", phi_m1)
np.save(OUT / "theta_m1_common_cohort.npy", theta_m1)

pd.DataFrame(
    phi_m1,
    columns=X_common.columns,
).to_csv(
    TABLE_DIR / "phi_m1_common_cohort.csv",
    index=False,
)

print("LDA warm start fitted:", phi_m1.shape)


## 6. Sparse M7b model with fixed or learned prior scales

For learned-scale runs:

- sigma_lithology ~ HalfNormal(0.5)
- sigma_age ~ HalfNormal(0.5)

The two scale hyperpriors are identical.

In [ ]:
class HGCTM_PriorSensitivity(nn.Module):
    def __init__(
        self,
        K,
        F,
        L,
        A,
        sigma_mu=2.0,
        sigma_litho=0.3,
        sigma_age=0.15,
        learn_scales=False,
        sigma_hyperprior_scale=0.5,
    ):
        super().__init__()
        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age
        self.learn_scales = learn_scales
        self.sigma_hyperprior_scale = sigma_hyperprior_scale

    def model(self, X, litho_idx, age_idx):
        N_local, F_local = X.shape
        K_local = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(K_local, F_local),
                self.sigma_mu
                * torch.ones(K_local, F_local),
            ).to_event(2),
        )

        if self.learn_scales:
            sigma_litho = pyro.sample(
                "sigma_litho",
                dist.HalfNormal(
                    torch.tensor(
                        self.sigma_hyperprior_scale,
                        dtype=X.dtype,
                    )
                ),
            )
            sigma_age = pyro.sample(
                "sigma_age",
                dist.HalfNormal(
                    torch.tensor(
                        self.sigma_hyperprior_scale,
                        dtype=X.dtype,
                    )
                ),
            )
        else:
            sigma_litho = torch.tensor(
                self.sigma_litho,
                dtype=X.dtype,
            )
            sigma_age = torch.tensor(
                self.sigma_age,
                dtype=X.dtype,
            )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(K_local, self.L, F_local),
                sigma_litho
                * torch.ones(K_local, self.L, F_local),
            ).to_event(3),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(K_local, self.A, F_local),
                sigma_age
                * torch.ones(K_local, self.A, F_local),
            ).to_event(3),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        B = pyro.sample(
            "B",
            dist.Normal(
                torch.zeros(self.L + self.A, K_local - 1),
                torch.ones(self.L + self.A, K_local - 1),
            ).to_event(2),
        )

        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                torch.zeros(K_local - 1),
                torch.ones(K_local - 1),
            ).to_event(1),
        )

        log_kappa = pyro.sample(
            "log_kappa",
            dist.Normal(3.5, 1.0),
        )
        kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N_local):
            z_litho = torch.nn.functional.one_hot(
                litho_idx,
                self.L,
            ).float()

            z_age = torch.nn.functional.one_hot(
                age_idx,
                self.A,
            ).float()

            z = torch.cat(
                [z_litho, z_age],
                dim=1,
            )

            eta = z @ B + intercept
            eta_full = torch.cat(
                [eta, torch.zeros(N_local, 1)],
                dim=1,
            )
            theta = softmax(eta_full, dim=1)

            litho_dev = delta_litho[
                :, litho_idx, :
            ].permute(1, 0, 2)

            age_dev = delta_age[
                :, age_idx, :
            ].permute(1, 0, 2)

            psi = (
                mu.unsqueeze(0)
                + tau.view(1, 1, -1) * litho_dev
                + omega_a
                * tau.view(1, 1, -1)
                * age_dev
            )

            phi = softmax(psi, dim=2)

            p = torch.einsum(
                "nk,nkf->nf",
                theta,
                phi,
            )
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            n_i = X.sum(dim=1)
            alpha = kappa * p

            pyro.sample(
                "obs",
                dist.DirichletMultinomial(
                    total_count=n_i.int(),
                    concentration=alpha,
                ),
                obs=X,
            )

    def guide(self, X, litho_idx, age_idx):
        _, F_local = X.shape
        K_local = self.K

        if self.learn_scales:
            sigma_litho_q_loc = pyro.param(
                "sigma_litho_q_loc",
                torch.tensor(math.log(0.20)),
            )
            sigma_litho_q_scale = pyro.param(
                "sigma_litho_q_scale",
                torch.tensor(0.25),
                constraint=torch.distributions.constraints.positive,
            )
            pyro.sample(
                "sigma_litho",
                dist.LogNormal(
                    sigma_litho_q_loc,
                    sigma_litho_q_scale,
                ),
            )

            sigma_age_q_loc = pyro.param(
                "sigma_age_q_loc",
                torch.tensor(math.log(0.20)),
            )
            sigma_age_q_scale = pyro.param(
                "sigma_age_q_scale",
                torch.tensor(0.25),
                constraint=torch.distributions.constraints.positive,
            )
            pyro.sample(
                "sigma_age",
                dist.LogNormal(
                    sigma_age_q_loc,
                    sigma_age_q_scale,
                ),
            )

        mu_loc = pyro.param(
            "mu_loc",
            torch.randn(K_local, F_local) * 0.1,
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5 * torch.ones(K_local, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "mu",
            dist.Normal(mu_loc, mu_scale).to_event(2),
        )

        delta_litho_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(K_local, self.L, F_local),
        )
        delta_litho_scale = pyro.param(
            "delta_litho_scale",
            0.2 * torch.ones(K_local, self.L, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(
                delta_litho_loc,
                delta_litho_scale,
            ).to_event(3),
        )

        delta_age_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(K_local, self.A, F_local),
        )
        delta_age_scale = pyro.param(
            "delta_age_scale",
            0.1 * torch.ones(K_local, self.A, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_age",
            dist.Normal(
                delta_age_loc,
                delta_age_scale,
            ).to_event(3),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "omega_a",
            dist.Beta(
                omega_a_a,
                omega_a_b,
            ),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "tau",
            dist.Beta(tau_a, tau_b).to_event(1),
        )

        B_loc = pyro.param(
            "B_loc",
            torch.zeros(self.L + self.A, K_local - 1),
        )
        B_scale = pyro.param(
            "B_scale",
            0.5 * torch.ones(self.L + self.A, K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "B",
            dist.Normal(B_loc, B_scale).to_event(2),
        )

        intercept_loc = pyro.param(
            "intercept_loc",
            torch.zeros(K_local - 1),
        )
        intercept_scale = pyro.param(
            "intercept_scale",
            0.5 * torch.ones(K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "intercept",
            dist.Normal(
                intercept_loc,
                intercept_scale,
            ).to_event(1),
        )

        log_kappa_loc = pyro.param(
            "log_kappa_loc",
            torch.tensor(3.5),
        )
        log_kappa_scale = pyro.param(
            "log_kappa_scale",
            torch.tensor(0.5),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "log_kappa",
            dist.Normal(
                log_kappa_loc,
                log_kappa_scale,
            ),
        )


## 7. Fitting and extraction utilities

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


def make_warm_mu_init(phi_reference):
    phi_clipped = np.clip(
        phi_reference,
        1e-6,
        None,
    )
    mu_init = np.log(phi_clipped)
    mu_init -= mu_init.mean(
        axis=1,
        keepdims=True,
    )
    return torch.tensor(
        mu_init,
        dtype=torch.float32,
    )


def fit_model(
    model,
    X,
    litho_idx,
    age_idx,
    phi_reference,
    seed,
):
    set_all_seeds(seed)
    pyro.clear_param_store()

    optimizer = ClippedAdam({
        "lr": LR,
        "betas": (0.9, 0.999),
    })

    svi = SVI(
        model.model,
        model.guide,
        optimizer,
        loss=Trace_ELBO(),
    )

    # Create parameters.
    svi.step(X, litho_idx, age_idx)

    # Exact warm-start logic used in the earlier sensitivity notebook.
    pyro.param("mu_loc").data.copy_(
        make_warm_mu_init(phi_reference)
    )

    pyro.param("tau_a").data.copy_(
        50.0 * torch.ones(X.shape[1])
    )
    pyro.param("tau_b").data.copy_(
        torch.ones(X.shape[1])
    )

    losses = []

    for step in range(N_STEPS):
        loss = svi.step(
            X,
            litho_idx,
            age_idx,
        )
        losses.append(float(loss))

        if step < ANNEAL_END:
            progress = step / ANNEAL_END
            minimum_tau_a = (
                50.0 * (1.0 - progress)
                + 1.0 * progress
            )
            with torch.no_grad():
                pyro.param("tau_a").data.clamp_(
                    min=minimum_tau_a
                )

        if (step + 1) % PRINT_EVERY == 0:
            tau_a = pyro.param(
                "tau_a"
            ).detach()
            tau_b = pyro.param(
                "tau_b"
            ).detach()
            tau_mean = (
                tau_a / (tau_a + tau_b)
            ).mean().item()

            message = (
                f"step {step+1:>5d} "
                f"loss={loss:>12.1f} "
                f"tau_mean={tau_mean:.3f}"
            )

            if model.learn_scales:
                litho_loc = pyro.param(
                    "sigma_litho_q_loc"
                ).item()
                litho_scale = pyro.param(
                    "sigma_litho_q_scale"
                ).item()
                age_loc = pyro.param(
                    "sigma_age_q_loc"
                ).item()
                age_scale = pyro.param(
                    "sigma_age_q_scale"
                ).item()

                litho_mean = math.exp(
                    litho_loc
                    + 0.5 * litho_scale ** 2
                )
                age_mean = math.exp(
                    age_loc
                    + 0.5 * age_scale ** 2
                )

                message += (
                    f" sigma_l={litho_mean:.3f}"
                    f" sigma_a={age_mean:.3f}"
                )

            print("   ", message)

    return losses


def extract_parameters(model):
    mu = pyro.param(
        "mu_loc"
    ).detach().cpu().numpy()

    delta_litho = pyro.param(
        "delta_litho_loc"
    ).detach().cpu().numpy()

    delta_age = pyro.param(
        "delta_age_loc"
    ).detach().cpu().numpy()

    B = pyro.param(
        "B_loc"
    ).detach().cpu().numpy()

    intercept = pyro.param(
        "intercept_loc"
    ).detach().cpu().numpy()

    tau_a = pyro.param(
        "tau_a"
    ).detach().cpu().numpy()
    tau_b = pyro.param(
        "tau_b"
    ).detach().cpu().numpy()
    tau = tau_a / (tau_a + tau_b)

    omega_a_a = float(
        pyro.param("omega_a_a").item()
    )
    omega_a_b = float(
        pyro.param("omega_a_b").item()
    )
    omega_a = omega_a_a / (
        omega_a_a + omega_a_b
    )

    kappa = float(
        np.exp(
            pyro.param("log_kappa_loc").item()
        )
    )

    phi_global = np.exp(mu)
    phi_global /= phi_global.sum(
        axis=1,
        keepdims=True,
    )

    if model.learn_scales:
        litho_q_loc = float(
            pyro.param(
                "sigma_litho_q_loc"
            ).item()
        )
        litho_q_scale = float(
            pyro.param(
                "sigma_litho_q_scale"
            ).item()
        )
        age_q_loc = float(
            pyro.param(
                "sigma_age_q_loc"
            ).item()
        )
        age_q_scale = float(
            pyro.param(
                "sigma_age_q_scale"
            ).item()
        )

        sigma_litho_mean = math.exp(
            litho_q_loc
            + 0.5 * litho_q_scale ** 2
        )
        sigma_age_mean = math.exp(
            age_q_loc
            + 0.5 * age_q_scale ** 2
        )
        sigma_litho_median = math.exp(
            litho_q_loc
        )
        sigma_age_median = math.exp(
            age_q_loc
        )
    else:
        sigma_litho_mean = float(
            model.sigma_litho
        )
        sigma_age_mean = float(
            model.sigma_age
        )
        sigma_litho_median = float(
            model.sigma_litho
        )
        sigma_age_median = float(
            model.sigma_age
        )
        litho_q_loc = np.nan
        litho_q_scale = np.nan
        age_q_loc = np.nan
        age_q_scale = np.nan

    return {
        "mu": mu,
        "phi_global": phi_global,
        "delta_litho": delta_litho,
        "delta_age": delta_age,
        "B": B,
        "intercept": intercept,
        "tau": tau,
        "omega_a": omega_a,
        "kappa": kappa,
        "sigma_litho_mean": sigma_litho_mean,
        "sigma_age_mean": sigma_age_mean,
        "sigma_litho_median": sigma_litho_median,
        "sigma_age_median": sigma_age_median,
        "sigma_litho_q_loc": litho_q_loc,
        "sigma_litho_q_scale": litho_q_scale,
        "sigma_age_q_loc": age_q_loc,
        "sigma_age_q_scale": age_q_scale,
    }


In [ ]:
def deterministic_outputs(
    params,
    X,
    litho_idx,
    age_idx,
):
    mu = torch.tensor(
        params["mu"],
        dtype=torch.float32,
    )
    delta_litho = torch.tensor(
        params["delta_litho"],
        dtype=torch.float32,
    )
    delta_age = torch.tensor(
        params["delta_age"],
        dtype=torch.float32,
    )
    B = torch.tensor(
        params["B"],
        dtype=torch.float32,
    )
    intercept = torch.tensor(
        params["intercept"],
        dtype=torch.float32,
    )
    tau = torch.tensor(
        params["tau"],
        dtype=torch.float32,
    )
    omega_a = torch.tensor(
        params["omega_a"],
        dtype=torch.float32,
    )

    N_local = X.shape[0]

    with torch.no_grad():
        z_litho = torch.nn.functional.one_hot(
            litho_idx,
            L,
        ).float()
        z_age = torch.nn.functional.one_hot(
            age_idx,
            A,
        ).float()
        z = torch.cat(
            [z_litho, z_age],
            dim=1,
        )

        eta = z @ B + intercept
        eta_full = torch.cat(
            [eta, torch.zeros(N_local, 1)],
            dim=1,
        )
        theta = softmax(
            eta_full,
            dim=1,
        )

        litho_dev = delta_litho[
            :, litho_idx, :
        ].permute(1, 0, 2)

        age_dev = delta_age[
            :, age_idx, :
        ].permute(1, 0, 2)

        psi = (
            mu.unsqueeze(0)
            + tau.view(1, 1, -1)
            * litho_dev
            + omega_a
            * tau.view(1, 1, -1)
            * age_dev
        )

        phi_context = softmax(
            psi,
            dim=2,
        )

        p = torch.einsum(
            "nk,nkf->nf",
            theta,
            phi_context,
        )
        p = p.clamp(min=1e-8)
        p /= p.sum(
            dim=1,
            keepdim=True,
        )

    return {
        "theta": theta.cpu().numpy(),
        "phi_context": phi_context.cpu().numpy(),
        "p": p.cpu().numpy(),
    }


def dirichlet_multinomial_log_prob(
    X,
    p,
    kappa,
):
    X_t = torch.tensor(
        X,
        dtype=torch.float64,
    )
    p_t = torch.tensor(
        p,
        dtype=torch.float64,
    )

    p_t = p_t.clamp(min=1e-12)
    p_t /= p_t.sum(
        dim=1,
        keepdim=True,
    )

    alpha = float(kappa) * p_t
    alpha0 = alpha.sum(dim=1)
    n_i = X_t.sum(dim=1)

    log_coefficient = (
        torch.lgamma(n_i + 1.0)
        - torch.lgamma(X_t + 1.0).sum(
            dim=1
        )
    )

    log_beta_ratio = (
        torch.lgamma(alpha0)
        - torch.lgamma(alpha0 + n_i)
        + (
            torch.lgamma(alpha + X_t)
            - torch.lgamma(alpha)
        ).sum(dim=1)
    )

    return (
        log_coefficient + log_beta_ratio
    ).cpu().numpy()


def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )
    if denominator == 0:
        return 0.0
    return float(
        np.dot(a, b) / denominator
    )


def align_topics_to_reference(
    phi_run,
    phi_reference,
):
    similarity = np.zeros((K, K))

    for run_topic in range(K):
        for reference_topic in range(K):
            similarity[
                run_topic,
                reference_topic,
            ] = cosine_similarity(
                phi_run[run_topic],
                phi_reference[reference_topic],
            )

    run_indices, reference_indices = (
        linear_sum_assignment(-similarity)
    )

    reference_to_run = np.full(
        K,
        -1,
        dtype=int,
    )

    for run_topic, reference_topic in zip(
        run_indices,
        reference_indices,
    ):
        reference_to_run[
            reference_topic
        ] = run_topic

    topic_cosines = np.array([
        similarity[
            reference_to_run[reference_topic],
            reference_topic,
        ]
        for reference_topic in range(K)
    ])

    return {
        "reference_to_run": reference_to_run,
        "aligned_phi": phi_run[
            reference_to_run
        ],
        "topic_cosines": topic_cosines,
        "similarity_matrix": similarity,
    }


## 8. Effect-size calculations

The notebook reports:

- observed-class unweighted effects;
- class-frequency-weighted effects across all observed categories;
- frequency-weighted nonmissing effects;
- frequency-weighted strict-resolved effects.

The **primary ratio** is the frequency-weighted nonmissing lithology effect divided by the frequency-weighted nonmissing age effect.

In [ ]:
def weighted_average(
    values,
    counts,
    include_mask,
):
    values = np.asarray(values, dtype=float)
    counts = np.asarray(counts, dtype=float)
    include_mask = np.asarray(
        include_mask,
        dtype=bool,
    )

    usable = (
        include_mask
        & (counts > 0)
        & np.isfinite(values)
    )

    if not usable.any():
        return np.nan

    return float(
        np.average(
            values[usable],
            weights=counts[usable],
        )
    )


def unweighted_average(
    values,
    counts,
    include_mask,
):
    values = np.asarray(values, dtype=float)
    counts = np.asarray(counts, dtype=float)
    include_mask = np.asarray(
        include_mask,
        dtype=bool,
    )

    usable = (
        include_mask
        & (counts > 0)
        & np.isfinite(values)
    )

    if not usable.any():
        return np.nan

    return float(
        values[usable].mean()
    )


def compute_effect_metrics(
    params,
    lithology_indices,
    age_indices,
):
    tau = params["tau"]

    lithology_class_effects = np.mean(
        np.abs(params["delta_litho"])
        * tau[None, None, :],
        axis=(0, 2),
    )

    age_class_effects = np.mean(
        np.abs(params["delta_age"])
        * tau[None, None, :]
        * params["omega_a"],
        axis=(0, 2),
    )

    lithology_counts = np.bincount(
        lithology_indices,
        minlength=L,
    )
    age_counts = np.bincount(
        age_indices,
        minlength=A,
    )

    lithology_observed = (
        lithology_counts > 0
    )
    age_observed = age_counts > 0

    lithology_nonmissing = np.array([
        label != "unknown"
        for label in all_model_levels
    ])
    age_nonmissing = np.array([
        label != "Unknown"
        for label in AGE_LEVELS
    ])

    lithology_strict = np.array([
        label not in {"unknown", "other"}
        for label in all_model_levels
    ])
    age_strict = age_nonmissing.copy()

    lithology_unweighted_observed = (
        unweighted_average(
            lithology_class_effects,
            lithology_counts,
            lithology_observed,
        )
    )
    age_unweighted_observed = (
        unweighted_average(
            age_class_effects,
            age_counts,
            age_observed,
        )
    )

    lithology_weighted_all = weighted_average(
        lithology_class_effects,
        lithology_counts,
        lithology_observed,
    )
    age_weighted_all = weighted_average(
        age_class_effects,
        age_counts,
        age_observed,
    )

    lithology_weighted_nonmissing = (
        weighted_average(
            lithology_class_effects,
            lithology_counts,
            lithology_nonmissing,
        )
    )
    age_weighted_nonmissing = (
        weighted_average(
            age_class_effects,
            age_counts,
            age_nonmissing,
        )
    )

    lithology_weighted_strict = (
        weighted_average(
            lithology_class_effects,
            lithology_counts,
            lithology_strict,
        )
    )
    age_weighted_strict = weighted_average(
        age_class_effects,
        age_counts,
        age_strict,
    )

    def safe_ratio(numerator, denominator):
        if (
            not np.isfinite(numerator)
            or not np.isfinite(denominator)
            or denominator <= 0
        ):
            return np.nan
        return float(numerator / denominator)

    metrics = {
        "lithology_effect_unweighted_observed":
            lithology_unweighted_observed,
        "age_effect_unweighted_observed":
            age_unweighted_observed,
        "ratio_unweighted_observed":
            safe_ratio(
                lithology_unweighted_observed,
                age_unweighted_observed,
            ),
        "lithology_effect_weighted_all":
            lithology_weighted_all,
        "age_effect_weighted_all":
            age_weighted_all,
        "ratio_weighted_all":
            safe_ratio(
                lithology_weighted_all,
                age_weighted_all,
            ),
        "lithology_effect_weighted_nonmissing":
            lithology_weighted_nonmissing,
        "age_effect_weighted_nonmissing":
            age_weighted_nonmissing,
        "ratio_weighted_nonmissing_primary":
            safe_ratio(
                lithology_weighted_nonmissing,
                age_weighted_nonmissing,
            ),
        "lithology_effect_weighted_strict":
            lithology_weighted_strict,
        "age_effect_weighted_strict":
            age_weighted_strict,
        "ratio_weighted_strict":
            safe_ratio(
                lithology_weighted_strict,
                age_weighted_strict,
            ),
        "tau_mean": float(np.mean(tau)),
        "tau_median": float(np.median(tau)),
    }

    detail_rows = []

    for index, label in enumerate(
        all_model_levels
    ):
        detail_rows.append({
            "covariate": "lithology",
            "class_index": index,
            "class_label": label,
            "count": int(lithology_counts[index]),
            "class_effect": float(
                lithology_class_effects[index]
            ),
            "is_missing_class": label == "unknown",
            "is_strict_resolved": (
                label not in {"unknown", "other"}
            ),
            "rare_under_threshold": (
                lithology_counts[index]
                < RARE_CLASS_THRESHOLD
            ),
        })

    for index, label in enumerate(AGE_LEVELS):
        detail_rows.append({
            "covariate": "age",
            "class_index": index,
            "class_label": label,
            "count": int(age_counts[index]),
            "class_effect": float(
                age_class_effects[index]
            ),
            "is_missing_class": label == "Unknown",
            "is_strict_resolved": label != "Unknown",
            "rare_under_threshold": (
                age_counts[index]
                < RARE_CLASS_THRESHOLD
            ),
        })

    return metrics, pd.DataFrame(detail_rows)


## 9. Run plan and resumable fitting

In [ ]:
run_plan_rows = []

for source_name, prior_names in RUN_PLAN.items():
    for prior_name in prior_names:
        for seed in ACTIVE_SEEDS:
            run_plan_rows.append({
                "source": source_name,
                "prior_setting": prior_name,
                "seed": seed,
                **PRIOR_SETTINGS[prior_name],
            })

run_plan_df = pd.DataFrame(run_plan_rows)
run_plan_df.to_csv(
    TABLE_DIR / "planned_runs.csv",
    index=False,
)

print(run_plan_df[
    [
        "source",
        "prior_setting",
        "seed",
        "learn_scales",
        "sigma_litho",
        "sigma_age",
    ]
].to_string(index=False))


In [ ]:
def safe_run_stem(
    source_name,
    prior_name,
    seed,
):
    return (
        f"{source_name.lower()}"
        f"__{prior_name}"
        f"__seed_{seed}"
    )


def load_existing_summary(run_stem):
    summary_path = (
        RUNS_DIR
        / f"{run_stem}_run_summary.json"
    )

    if not summary_path.exists():
        return None

    with open(
        summary_path,
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def save_run(
    run_stem,
    source_name,
    prior_name,
    seed,
    losses,
    params,
    outputs,
    alignment,
    effect_metrics,
    effect_details,
    log_prob_per_document,
    elapsed_seconds,
):
    archive_path = (
        RUNS_DIR / f"{run_stem}.npz"
    )

    np.savez_compressed(
        archive_path,
        source=np.array(source_name),
        prior_setting=np.array(prior_name),
        seed=np.array(seed),
        Mindat_id=np.array(common_ids),
        family_names=np.array(
            X_common.columns
        ),
        lithology_levels=np.array(
            all_model_levels
        ),
        age_levels=np.array(AGE_LEVELS),
        losses=np.array(losses),
        phi_global_raw=params["phi_global"],
        phi_global_aligned=alignment[
            "aligned_phi"
        ],
        mu=params["mu"],
        delta_litho=params[
            "delta_litho"
        ],
        delta_age=params[
            "delta_age"
        ],
        B=params["B"],
        intercept=params["intercept"],
        tau=params["tau"],
        omega_a=np.array(
            params["omega_a"]
        ),
        kappa=np.array(
            params["kappa"]
        ),
        sigma_litho_mean=np.array(
            params["sigma_litho_mean"]
        ),
        sigma_age_mean=np.array(
            params["sigma_age_mean"]
        ),
        theta=outputs["theta"],
        fitted_family_probabilities=outputs["p"],
        dm_log_prob_per_document=(
            log_prob_per_document
        ),
        reference_to_run=alignment[
            "reference_to_run"
        ],
        topic_cosines_to_lda=alignment[
            "topic_cosines"
        ],
    )

    pd.DataFrame({
        "step": np.arange(
            1,
            len(losses) + 1,
        ),
        "loss": losses,
        "elbo": -np.asarray(losses),
    }).to_csv(
        RUNS_DIR
        / f"{run_stem}_loss_trace.csv",
        index=False,
    )

    effect_details.to_csv(
        RUNS_DIR
        / f"{run_stem}_class_effects.csv",
        index=False,
    )

    total_recorded_counts = float(
        X_np.sum()
    )
    document_counts = X_np.sum(axis=1)

    summary = {
        "run_stem": run_stem,
        "source": source_name,
        "prior_setting": prior_name,
        "seed": int(seed),
        "run_mode": RUN_MODE,
        "n_deposits": int(N),
        "n_families": int(F),
        "n_steps": int(len(losses)),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "learn_scales": bool(
            PRIOR_SETTINGS[
                prior_name
            ]["learn_scales"]
        ),
        "fixed_sigma_litho": (
            PRIOR_SETTINGS[
                prior_name
            ]["sigma_litho"]
        ),
        "fixed_sigma_age": (
            PRIOR_SETTINGS[
                prior_name
            ]["sigma_age"]
        ),
        "posterior_sigma_litho_mean": float(
            params["sigma_litho_mean"]
        ),
        "posterior_sigma_age_mean": float(
            params["sigma_age_mean"]
        ),
        "posterior_sigma_litho_median": float(
            params["sigma_litho_median"]
        ),
        "posterior_sigma_age_median": float(
            params["sigma_age_median"]
        ),
        "posterior_sigma_ratio_lithology_over_age": float(
            params["sigma_litho_mean"]
            / max(
                params["sigma_age_mean"],
                1e-12,
            )
        ),
        "omega_a": float(
            params["omega_a"]
        ),
        "effective_age_scale_sigma_times_omega": float(
            params["sigma_age_mean"]
            * params["omega_a"]
        ),
        "kappa": float(params["kappa"]),
        "elbo_last": float(
            -losses[-1]
        ),
        "elbo_mean_last_100": float(
            -np.mean(losses[-100:])
        ),
        "elbo_mean_last_100_per_deposit": float(
            -np.mean(losses[-100:])
            / N
        ),
        "dm_log_prob_total": float(
            log_prob_per_document.sum()
        ),
        "dm_log_prob_mean_per_deposit": float(
            log_prob_per_document.mean()
        ),
        "dm_log_prob_total_per_recorded_count": float(
            log_prob_per_document.sum()
            / total_recorded_counts
        ),
        "mean_document_log_prob_per_recorded_count": float(
            np.mean(
                log_prob_per_document
                / np.maximum(
                    document_counts,
                    1,
                )
            )
        ),
        "mean_topic_cosine_to_common_lda": float(
            alignment[
                "topic_cosines"
            ].mean()
        ),
        "minimum_topic_cosine_to_common_lda": float(
            alignment[
                "topic_cosines"
            ].min()
        ),
        **{
            key: (
                float(value)
                if np.isfinite(value)
                else None
            )
            for key, value
            in effect_metrics.items()
        },
    }

    with open(
        RUNS_DIR
        / f"{run_stem}_run_summary.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    return summary


In [ ]:
summaries = []

for source_name, prior_names in RUN_PLAN.items():
    lithology_indices = (
        source_lithology_idx[source_name]
    )
    lithology_tensor = (
        source_lithology_tensors[source_name]
    )

    for prior_name in prior_names:
        prior = PRIOR_SETTINGS[prior_name]

        for seed in ACTIVE_SEEDS:
            run_stem = safe_run_stem(
                source_name,
                prior_name,
                seed,
            )

            existing = load_existing_summary(
                run_stem
            )

            if existing is not None:
                print(
                    f"SKIP completed run: {run_stem}"
                )
                summaries.append(existing)
                continue

            print("\n" + "=" * 84)
            print(
                f"SOURCE={source_name} | "
                f"PRIOR={prior_name} | "
                f"SEED={seed}"
            )
            print("=" * 84)

            model = HGCTM_PriorSensitivity(
                K=K,
                F=F,
                L=L,
                A=A,
                sigma_mu=2.0,
                sigma_litho=(
                    prior["sigma_litho"]
                    if not prior["learn_scales"]
                    else 0.3
                ),
                sigma_age=(
                    prior["sigma_age"]
                    if not prior["learn_scales"]
                    else 0.15
                ),
                learn_scales=prior[
                    "learn_scales"
                ],
                sigma_hyperprior_scale=(
                    SIGMA_HYPERPRIOR_SCALE
                ),
            )

            start = time.time()

            losses = fit_model(
                model=model,
                X=X_tensor,
                litho_idx=lithology_tensor,
                age_idx=age_tensor,
                phi_reference=phi_m1,
                seed=seed,
            )

            elapsed = time.time() - start

            params = extract_parameters(
                model
            )

            outputs = deterministic_outputs(
                params,
                X_tensor,
                lithology_tensor,
                age_tensor,
            )

            log_prob_per_document = (
                dirichlet_multinomial_log_prob(
                    X_np,
                    outputs["p"],
                    params["kappa"],
                )
            )

            alignment = align_topics_to_reference(
                params["phi_global"],
                phi_m1,
            )

            effect_metrics, effect_details = (
                compute_effect_metrics(
                    params,
                    lithology_indices,
                    age_idx_np,
                )
            )

            summary = save_run(
                run_stem=run_stem,
                source_name=source_name,
                prior_name=prior_name,
                seed=seed,
                losses=losses,
                params=params,
                outputs=outputs,
                alignment=alignment,
                effect_metrics=effect_metrics,
                effect_details=effect_details,
                log_prob_per_document=(
                    log_prob_per_document
                ),
                elapsed_seconds=elapsed,
            )

            summaries.append(summary)

            print(
                f"Completed {run_stem}: "
                f"primary ratio="
                f"{summary['ratio_weighted_nonmissing_primary']:.3f}; "
                f"strict ratio="
                f"{summary['ratio_weighted_strict']:.3f}; "
                f"topic cosine="
                f"{summary['mean_topic_cosine_to_common_lda']:.3f}"
            )

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(
    TABLE_DIR / "all_run_summary.csv",
    index=False,
)

print("\nCompleted runs:", len(summary_df))
print(
    summary_df[
        [
            "source",
            "prior_setting",
            "seed",
            "ratio_weighted_nonmissing_primary",
            "ratio_weighted_strict",
            "posterior_sigma_litho_mean",
            "posterior_sigma_age_mean",
            "omega_a",
            "mean_topic_cosine_to_common_lda",
        ]
    ].round(4).to_string(index=False)
)


## 10. Aggregate results across seeds

In [ ]:
numeric_metrics = [
    "ratio_weighted_nonmissing_primary",
    "ratio_weighted_all",
    "ratio_weighted_strict",
    "ratio_unweighted_observed",
    "lithology_effect_weighted_nonmissing",
    "age_effect_weighted_nonmissing",
    "posterior_sigma_litho_mean",
    "posterior_sigma_age_mean",
    "posterior_sigma_ratio_lithology_over_age",
    "effective_age_scale_sigma_times_omega",
    "omega_a",
    "kappa",
    "elbo_mean_last_100_per_deposit",
    "dm_log_prob_mean_per_deposit",
    "dm_log_prob_total_per_recorded_count",
    "mean_document_log_prob_per_recorded_count",
    "mean_topic_cosine_to_common_lda",
    "minimum_topic_cosine_to_common_lda",
]

aggregate_rows = []

for (source_name, prior_name), group in summary_df.groupby(
    ["source", "prior_setting"],
    sort=True,
):
    row = {
        "source": source_name,
        "prior_setting": prior_name,
        "n_seeds": int(len(group)),
    }

    for metric in numeric_metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce",
        )
        row[f"{metric}_mean"] = float(
            values.mean()
        )
        row[f"{metric}_median"] = float(
            values.median()
        )
        row[f"{metric}_min"] = float(
            values.min()
        )
        row[f"{metric}_max"] = float(
            values.max()
        )
        row[f"{metric}_std"] = float(
            values.std(ddof=0)
        )

    primary_ratio = pd.to_numeric(
        group[
            "ratio_weighted_nonmissing_primary"
        ],
        errors="coerce",
    )
    strict_ratio = pd.to_numeric(
        group[
            "ratio_weighted_strict"
        ],
        errors="coerce",
    )

    row["primary_ratio_all_seeds_gt_1"] = bool(
        (primary_ratio > 1).all()
    )
    row["strict_ratio_all_seeds_gt_1"] = bool(
        (strict_ratio > 1).all()
    )

    aggregate_rows.append(row)

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR
    / "configuration_summary_across_seeds.csv",
    index=False,
)

display_columns = [
    "source",
    "prior_setting",
    "n_seeds",
    "ratio_weighted_nonmissing_primary_median",
    "ratio_weighted_nonmissing_primary_min",
    "ratio_weighted_nonmissing_primary_max",
    "ratio_weighted_strict_median",
    "posterior_sigma_litho_mean_median",
    "posterior_sigma_age_mean_median",
    "omega_a_median",
    "mean_topic_cosine_to_common_lda_median",
    "primary_ratio_all_seeds_gt_1",
]

print(
    aggregate_df[
        display_columns
    ].round(4).to_string(index=False)
)


## 11. Topic stability across seeds and sources

In [ ]:
def load_run_npz(
    source_name,
    prior_name,
    seed,
):
    run_stem = safe_run_stem(
        source_name,
        prior_name,
        seed,
    )
    return np.load(
        RUNS_DIR / f"{run_stem}.npz",
        allow_pickle=True,
    )


within_seed_rows = []

for source_name, prior_names in RUN_PLAN.items():
    for prior_name in prior_names:
        available_seeds = [
            seed
            for seed in ACTIVE_SEEDS
            if (
                RUNS_DIR
                / f"{safe_run_stem(source_name, prior_name, seed)}.npz"
            ).exists()
        ]

        if not available_seeds:
            continue

        reference_seed = (
            PRIMARY_SEED
            if PRIMARY_SEED in available_seeds
            else available_seeds[0]
        )
        reference = load_run_npz(
            source_name,
            prior_name,
            reference_seed,
        )["phi_global_aligned"]

        for seed in available_seeds:
            current = load_run_npz(
                source_name,
                prior_name,
                seed,
            )["phi_global_aligned"]

            for topic in range(K):
                within_seed_rows.append({
                    "source": source_name,
                    "prior_setting": prior_name,
                    "reference_seed": reference_seed,
                    "seed": seed,
                    "topic": topic + 1,
                    "cosine_to_reference_seed": (
                        cosine_similarity(
                            current[topic],
                            reference[topic],
                        )
                    ),
                })

within_seed_stability = pd.DataFrame(
    within_seed_rows
)
within_seed_stability.to_csv(
    TABLE_DIR / "within_configuration_seed_stability.csv",
    index=False,
)

cross_source_rows = []

common_prior_names = sorted(
    set.intersection(
        *[
            set(priors)
            for priors in RUN_PLAN.values()
        ]
    )
)

for prior_name in common_prior_names:
    sources = list(RUN_PLAN)

    for source_1 in sources:
        for source_2 in sources:
            path_1 = (
                RUNS_DIR
                / f"{safe_run_stem(source_1, prior_name, PRIMARY_SEED)}.npz"
            )
            path_2 = (
                RUNS_DIR
                / f"{safe_run_stem(source_2, prior_name, PRIMARY_SEED)}.npz"
            )

            if not (
                path_1.exists()
                and path_2.exists()
            ):
                continue

            phi_1 = np.load(
                path_1,
                allow_pickle=True,
            )["phi_global_aligned"]

            phi_2 = np.load(
                path_2,
                allow_pickle=True,
            )["phi_global_aligned"]

            topic_cosines = [
                cosine_similarity(
                    phi_1[topic],
                    phi_2[topic],
                )
                for topic in range(K)
            ]

            cross_source_rows.append({
                "prior_setting": prior_name,
                "source_1": source_1,
                "source_2": source_2,
                "mean_topic_cosine": float(
                    np.mean(topic_cosines)
                ),
                "minimum_topic_cosine": float(
                    np.min(topic_cosines)
                ),
                **{
                    f"topic_{topic+1}_cosine":
                        topic_cosines[topic]
                    for topic in range(K)
                },
            })

cross_source_alignment = pd.DataFrame(
    cross_source_rows
)
cross_source_alignment.to_csv(
    TABLE_DIR / "between_source_topic_alignment_by_prior.csv",
    index=False,
)

if len(within_seed_stability):
    print("Within-configuration seed stability:")
    print(
        within_seed_stability.groupby(
            ["source", "prior_setting"]
        )["cosine_to_reference_seed"].agg(
            ["mean", "min", "std"]
        ).round(4).to_string()
    )

if len(cross_source_alignment):
    print("\nBetween-source alignment:")
    print(
        cross_source_alignment[
            [
                "prior_setting",
                "source_1",
                "source_2",
                "mean_topic_cosine",
                "minimum_topic_cosine",
            ]
        ].round(4).to_string(index=False)
    )


## 12. Decision table for reviewer interpretation

In [ ]:
decision_rows = []

for _, row in aggregate_df.iterrows():
    source_name = row["source"]
    prior_name = row["prior_setting"]

    ratio_min = row[
        "ratio_weighted_nonmissing_primary_min"
    ]
    ratio_median = row[
        "ratio_weighted_nonmissing_primary_median"
    ]
    strict_min = row[
        "ratio_weighted_strict_min"
    ]
    topic_min = row[
        "mean_topic_cosine_to_common_lda_min"
    ]

    if ratio_min > 1 and strict_min > 1:
        result_flag = "lithology_stronger_all_seeds"
    elif ratio_median > 1:
        result_flag = "lithology_stronger_median_but_not_all_checks"
    else:
        result_flag = "lithology_not_consistently_stronger"

    decision_rows.append({
        "source": source_name,
        "prior_setting": prior_name,
        "primary_ratio_median": ratio_median,
        "primary_ratio_min": ratio_min,
        "strict_ratio_min": strict_min,
        "topic_alignment_minimum_seed_mean": topic_min,
        "result_flag": result_flag,
        "reviewer_relevance": (
            "Direct prior-sensitivity evidence"
            if prior_name in {
                "equal_low",
                "equal_high",
                "reversed",
                "learned_equal_hyperprior",
            }
            else "Submitted-prior reference"
        ),
    })

decision_table = pd.DataFrame(
    decision_rows
)
decision_table.to_csv(
    TABLE_DIR / "reviewer_decision_table.csv",
    index=False,
)

print(decision_table.round(4).to_string(index=False))


## 13. Figures

In [ ]:
# Figure 1: primary weighted ratio across all runs.
plot_df = summary_df.copy()

fig, ax = plt.subplots(figsize=(13, 6))

x_labels = [
    f"{source}\n{prior}"
    for source, prior in zip(
        plot_df["source"],
        plot_df["prior_setting"],
    )
]

x_positions = np.arange(len(plot_df))

ax.scatter(
    x_positions,
    plot_df[
        "ratio_weighted_nonmissing_primary"
    ],
)
ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)
ax.set_xticks(x_positions)
ax.set_xticklabels(
    x_labels,
    rotation=70,
    ha="right",
)
ax.set_ylabel(
    "Frequency-weighted lithology / age effect ratio"
)
ax.set_title(
    "Prior and lithology-source sensitivity across seeds"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "primary_weighted_ratio_all_runs.png",
    dpi=300,
)
plt.close()


# Figure 2: median ratio by configuration.
median_plot = aggregate_df.copy()
median_plot["label"] = (
    median_plot["source"]
    + "\n"
    + median_plot["prior_setting"]
)

fig, ax = plt.subplots(figsize=(12, 6))

positions = np.arange(
    len(median_plot)
)

ax.errorbar(
    positions,
    median_plot[
        "ratio_weighted_nonmissing_primary_median"
    ],
    yerr=[
        median_plot[
            "ratio_weighted_nonmissing_primary_median"
        ]
        - median_plot[
            "ratio_weighted_nonmissing_primary_min"
        ],
        median_plot[
            "ratio_weighted_nonmissing_primary_max"
        ]
        - median_plot[
            "ratio_weighted_nonmissing_primary_median"
        ],
    ],
    fmt="o",
    capsize=4,
)
ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)
ax.set_xticks(positions)
ax.set_xticklabels(
    median_plot["label"],
    rotation=65,
    ha="right",
)
ax.set_ylabel(
    "Median primary ratio with seed range"
)
ax.set_title(
    "Configuration-level lithology-to-age sensitivity"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "configuration_ratio_median_and_range.png",
    dpi=300,
)
plt.close()


# Figure 3: learned posterior scales.
learned = summary_df[
    summary_df["learn_scales"].eq(True)
].copy()

if len(learned):
    fig, ax = plt.subplots(figsize=(9, 5))
    positions = np.arange(len(learned))
    width = 0.35

    ax.bar(
        positions - width / 2,
        learned[
            "posterior_sigma_litho_mean"
        ],
        width,
        label="Lithology",
    )
    ax.bar(
        positions + width / 2,
        learned[
            "posterior_sigma_age_mean"
        ],
        width,
        label="Age",
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(
        [
            f"{source}\nseed {seed}"
            for source, seed in zip(
                learned["source"],
                learned["seed"],
            )
        ],
        rotation=35,
        ha="right",
    )
    ax.set_ylabel("Posterior mean scale")
    ax.set_title(
        "Learned lithology and age scales under equal hyperpriors"
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        FIG_DIR
        / "learned_scale_posterior_means.png",
        dpi=300,
    )
    plt.close()

print("Saved figures:")
for path in sorted(FIG_DIR.iterdir()):
    print(" -", path.name)


## 14. Provenance and downloadable archive

In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "HGCTM_Prior_and_Lithology_Class_Sensitivity_K7.ipynb",
    "original_notebook_sha256": "6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b",
    "run_mode": RUN_MODE,
    "stage0_directory": str(STAGE0),
    "phase1b_directory": str(PHASE1B),
    "cohort": {
        "original_model_n": int(X_model.shape[0]),
        "excluded_invalid_coordinates": int(len(excluded_invalid)),
        "common_valid_coordinate_n": int(N),
        "mineral_families": int(F),
    },
    "class_preparation": {
        "raw_labels_preserved": True,
        "evaporite_model_recode": "other",
        "rare_class_warning_threshold": RARE_CLASS_THRESHOLD,
        "unknown_retained_as_missingness_category": True,
    },
    "model": {
        "K": K,
        "sigma_mu": 2.0,
        "likelihood": "Dirichlet-Multinomial",
        "tau_prior": "Beta(1,3)",
        "omega_age_prior": "Beta(2,2)",
        "learning_rate": LR,
        "steps": N_STEPS,
        "anneal_end": ANNEAL_END,
        "seeds": ACTIVE_SEEDS,
        "LDA_warm_start_random_state": 42,
    },
    "prior_settings": PRIOR_SETTINGS,
    "run_plan": RUN_PLAN,
    "learned_scale_model": {
        "sigma_lithology_hyperprior": (
            f"HalfNormal({SIGMA_HYPERPRIOR_SCALE})"
        ),
        "sigma_age_hyperprior": (
            f"HalfNormal({SIGMA_HYPERPRIOR_SCALE})"
        ),
        "variational_family": "LogNormal",
        "interpretation_note": (
            "Age scale and omega_a jointly determine effective age deviations."
        ),
    },
    "primary_effect_metric": (
        "frequency-weighted nonmissing lithology effect divided by "
        "frequency-weighted nonmissing age effect"
    ),
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

environment = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "pyro": pyro.__version__,
}

with open(
    OUT / "environment_versions.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        environment,
        handle,
        indent=2,
    )

readme = '''
HGCTM prior and lithology-class sensitivity at fixed K=7

This archive contains:
- 13 source/prior configurations;
- one or three seeds according to RUN_MODE;
- resumable per-run parameter archives and loss traces;
- fixed-scale and learned-scale results;
- frequency-weighted, unweighted, nonmissing, and strict-resolved effects;
- likelihood per deposit and per recorded mineral count;
- topic alignment and seed stability;
- rare-class warnings;
- reviewer-oriented decision tables.

For model fitting, the single raw GLiM evaporite observation is merged
into the broad other class. The raw class remains preserved in the audit
table.

The primary ratio is the class-frequency-weighted nonmissing lithology
effect divided by the corresponding nonmissing age effect.

The learned-scale model uses identical HalfNormal(0.5) hyperpriors for
lithology and age. Because the original omega_a parameter is retained,
the learned age scale must be interpreted jointly with omega_a.
'''.strip()

(OUT / "README.txt").write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(
        WORK
        / "HGCTM_Prior_and_Lithology_Class_Sensitivity_K7_Outputs"
    ),
    "zip",
    root_dir=OUT,
)

print("Output archive:")
print(archive_path)

print("\nKey tables:")
for path in sorted(TABLE_DIR.iterdir()):
    print(" -", path.name)
